# Deepfakes & Misuse— Defending Synthetic Media

**Module 15 · Lesson 15.5**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- The Threat Surface
- The Detection Arms Race
- Provenance (C2PA / Content Credentials)
- Watermarking
- The Misuse Risk Matrix
- Defense-in-Depth

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

## You cannot spot every fake - so stop trying to

> 💡 **Info**
>
> The reflex when facing deepfakes is to build a better detector. It is the wrong reflex, and understanding why is the key to this whole lesson. Detection is an arms race you lose: every artifact a detector learns to spot, the next generator learns to erase, so accuracy decays month over month. And even a good detector is defeated by arithmetic — when real media vastly outnumbers fake, a small false-positive rate floods you with false alarms. So the strategy flips: instead of trying to prove what is *fake*, you prove what is *real* at the source with cryptographic provenance, you watermark your own outputs so they carry a signal, you assess which misuses your capabilities enable and rank them by risk, you stack independent guardrails so no single failure is fatal, and you disclose what is AI-generated because the law now requires it. This lesson builds that toolkit as engineering checks: a threat-surface map, a precision calculation that shows why detection loses, a provenance-chain validator, a watermark detector and its removal attack, a misuse risk matrix, a defense-in-depth stack, and a disclosure gate tied to the EU AI Act. It will not make you a content-moderation team; it will make you the engineer who ships a generative feature with the misuse thought through.

### 🎯 What you will be able to do after this lesson

- **Apply** a threat-surface map and a misuse risk matrix to rank which harms a generative product enables.

- **Analyze** why deepfake detection loses to better generators and to the base-rate problem, and why provenance is the durable answer.

- **Evaluate** a C2PA provenance chain and a statistical watermark, including their failure modes.

- **Create** a defense-in-depth stack and a disclosure gate that meets the EU AI Act Article 50 transparency obligation.

> 📦 **Info**
>
> ✅ Before you startThis lesson follows **15.4 (Privacy)**: privacy protects the real person’s data; a deepfake fabricates a *synthetic* version of a person — their face, voice, likeness — so it is a misuse of identity rather than a leak of data, though the two overlap. It picks up **C2PA provenance**, hooked forward from **15.3 (Copyright)** as the way to prove how media was made, and it reuses the **defense-in-depth** idea from **Module 12** (independent layers whose failures multiply), now applied to abuse. Everything here is keyless arithmetic, and it is defensive engineering education.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🎭 **Analogy**
>
> Synthetic media is **counterfeit money the eye can no longer catch**. For most of history a careful look told a real bill from a fake; then printing got good enough that the naked eye lost, and the response was not “look harder” — it was to build verification *into* the currency: watermarks, threads, holograms, serial numbers you can check against a source. Deepfakes are counterfeit faces, voices, and footage that have crossed the same threshold: staring at the pixels no longer works. So the defense is the same shift the mint made — stop trusting the eye, and build verifiable authenticity into the media itself. **Where it breaks down:** a banknote is issued by one central mint, while anyone can generate media, so authenticity has to be signed at creation by many sources — which is exactly the provenance problem this lesson gets to.

> 📦 **Info**
>
> 🚫 Three misconceptions this lesson kills
> - **“We will just detect the fakes.”** Detection decays against better generators and floods false positives at low base rates. You cannot detect your way out.
> - **“A watermark makes content tamper-proof.”** Paraphrasing, cropping, or re-encoding can remove it. A watermark is a strong signal, not a guarantee.
> - **“Provenance proves something is fake.”** It proves what is *real*; a missing manifest is a signal, not proof, and it only helps once signing is widespread.

> 💡 **Info**
>
> ⚠️ Anti-patternShipping a generative capability with no misuse assessment, no output provenance or watermark, and no takedown path — then reacting only after an incident. By the time the abuse report arrives, a victim has already been harmed, there is no disclosure label to point to, no process to pull the content, and no way to respond within the law. The whole lesson is the opposite discipline: decide before launch which harms you enable, make your outputs verifiable, layer the guardrails, and have the disclosure and the takedown path ready on day one.

---

## 🎯 Concept 1: The Threat Surface

### The Threat Surface

Generative models make convincing fakes across text, image, audio, and video, and YOUR capabilities decide which harms you enable. Map each harm to the modality it needs - non-consensual imagery (image), voice-clone fraud (audio), disinformation (image/video), impersonation (image+audio), fabricated evidence, blackmail - and adding a modality widens the surface. Know what you enable before you ship.

Start by naming what can go wrong, because you cannot defend a threat you have not mapped. Synthetic-media harms fall into recognizable categories: **non-consensual intimate imagery** (the most-reported deepfake abuse), **voice-clone fraud** (the fastest-growing — the fake-CEO wire-transfer scam and the “family emergency” call), **political disinformation**, **impersonation** that defeats face or voice identity checks, **fabricated evidence**, and **deepfake-video coercion**. The key insight for an engineer is that your *threat surface is defined by what your product can generate*: each harm needs a certain modality, so a product that makes text, image, and audio — but not video — enables five of the six categories in the worked example, and adding video would unlock the last one. This is not abstract hand-wringing; it is a concrete inventory you produce before launch, the same way you would enumerate a system’s attack surface before a security review. You cannot mitigate what you have not listed. And the very first control on that surface is **consent**: for any harm that impersonates a specific real person, the front-door defense is to verify you are authorized to synthesize that identity at all — enrolling a voice only from its verified owner, blocklisting public figures and known individuals, and refusing to clone a face or voice on demand. A synthesis you never allowed is one you never have to detect, watermark, or take down; consent-to-synthesize (now backed by likeness and voice-rights laws such as the US NO FAKES proposals) is the cheapest control you have. The block maps capabilities to enabled harms, keyless.

> 💰 **Analogy**
>
> Your threat surface is **the set of doors a master key opens**. A building’s security team does not ask “is this key dangerous?” in the abstract; they ask *which specific doors it opens* — the supply closet, the server room, the vault — because that list is the exposure. A generative model is a master key, and each capability (text, image, audio, video) opens a different set of harm-doors. Enumerate them: an image model opens the non-consensual-imagery door and the fake-evidence door; an audio model opens the voice-fraud door. You do not secure a building by worrying that keys exist; you secure it by knowing exactly which doors yours opens, and putting a guard on each.

Your product generates text, image, and audio, but not video. How many of the six harm categories does it enable?

**📝 `01_threat_surface.py`** - *runnable*

In [ ]:
# THE THREAT SURFACE: generative models make convincing fakes across modalities, and YOUR capabilities decide which harms you
# enable. Map each harm to the modality it needs, then see which ones your product unlocks. Adding a modality widens the surface.
HARMS = [  # (harm, modalities it needs (any-of), the real-world harm)
    ("non-consensual intimate imagery", {"image"},          "the most-reported deepfake abuse"),
    ("voice-clone fraud (vishing)",     {"audio"},          "fastest-growing: fake-CEO and family-emergency scams"),
    ("political disinformation",        {"image", "video"}, "erodes trust; election interference"),
    ("impersonation / KYC bypass",      {"image", "audio"}, "defeats face and voice identity checks"),
    ("fabricated evidence",             {"image", "video"}, "fake documents, fake footage"),
    ("deepfake-video blackmail",        {"video"},          "coercion with fabricated video")]
product_caps = {"text", "image", "audio"}       # our product generates these; NOT video
print("Product capabilities: {}".format(sorted(product_caps)))
enabled = 0
for harm, needs, why in HARMS:
    ok = bool(needs & product_caps)             # enabled if we can produce ANY modality the harm needs
    print("  [{}] {:<34} needs {:<16} {}".format("X" if ok else " ", harm, "/".join(sorted(needs)), why))
    if ok: enabled += 1
print()
print("{} of {} harm categories are ENABLED by this product's capabilities.".format(enabled, len(HARMS)))
print("Your threat surface is defined by what you can generate - adding video would unlock the last one. Map it before you ship.")

# Output:
# Product capabilities: ['audio', 'image', 'text']
#   [X] non-consensual intimate imagery    needs image            the most-reported deepfake abuse
#   [X] voice-clone fraud (vishing)        needs audio            fastest-growing: fake-CEO and family-emergency scams
#   [X] political disinformation           needs image/video      erodes trust; election interference
#   [X] impersonation / KYC bypass         needs audio/image      defeats face and voice identity checks
#   [X] fabricated evidence                needs image/video      fake documents, fake footage
#   [ ] deepfake-video blackmail           needs video            coercion with fabricated video
#
# 5 of 6 harm categories are ENABLED by this product's capabilities.
# Your threat surface is defined by what you can generate - adding video would unlock the last one. Map it before you ship.

- Each harm is tagged with the modality it needs; a harm is **enabled** if you can generate any modality it needs.
- The product generates text, image, and audio — so it enables **5 of 6** harms.
- Only deepfake-video blackmail (video-only) is out of reach — until you add video, which would unlock the rest.
- Your threat surface is your capability list; enumerate it before launch, the way you would a security attack surface.

#### 💡 What just happened

⚡ What just happened? Your generative capabilities define your threat surface: map each harm to the modality it needs and you find this product enables 5 of 6. Tradeoff: none - it is an inventory you owe before shipping. Next: the instinct is to detect the fakes you might produce - and that is the losing move.

- A harm map where a product’s capabilities light up the harms it enables
- Product text+image+audio enables 5 of 6 harms

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: The Detection Arms Race

### The Detection Arms Race

The reflex is to run a detector. It loses twice. First, detectors chase generator artifacts, so a better generator erases them and accuracy decays (95 percent on an old generator, 62 percent on a new one). Second, at a realistic base rate - few fakes among many real files - even a good detector floods false positives. You cannot detect your way out; prove what is real instead.

Faced with fakes, almost everyone reaches for a detector — a classifier that flags synthetic media. It is a supporting tool, but it cannot be your strategy, and it fails in two independent ways. First, the **arms race**: detectors work by spotting artifacts (unnatural blinking, frequency signatures, texture glitches), and every one of those is a target the next generator is trained to eliminate. So the *same* detector that scored 95 percent on a 2023 generator scores 62 percent on a 2026 one — accuracy decays as the generators improve, and the defender is always a step behind. Second, and less intuitively, the **base-rate problem**: in the real world the vast majority of media is authentic, so even a small false-positive rate produces a flood of false alarms. In the worked example, among 10,000 items with a 1 percent fake rate, a detector with a 90 percent true-positive and 10 percent false-positive rate flags 90 real fakes and 990 authentic items — a **precision of about 8 percent**, meaning 92 percent of everything it flags is a false alarm. A tool that cries wolf nine times out of ten is unusable as a gatekeeper. Together these say: you cannot detect your way out. The durable move is to stop chasing fakes and start proving what is real — which is the next step. The block computes the decay and the base-rate precision, keyless.

> ⚔️ **Analogy**
>
> Deepfake detection is a **lock-picking race the locksmith keeps losing**. Every clever new lock buys a few months until the pickers catch up, and then you are back to square one, having spent real effort to end up even. Worse, imagine a guard told to stop “anyone suspicious” at a stadium where almost everyone is innocent: set the bar low enough to catch the rare bad actor and you detain hundreds of ordinary fans, and the security line collapses under false alarms. Detection is both races at once — forever behind the generators, and drowning in false positives against a crowd of real media. The winning move is not a better lock; it is a key that proves who belongs.

You will defend against deepfakes by running a detector on incoming media. Why is that not enough?

**📝 `02_detection_arms_race.py`** - *runnable*

In [ ]:
# THE DETECTION ARMS RACE + THE BASE-RATE PROBLEM: deepfake detectors chase artifacts, and every better generator erases them,
# so accuracy decays. Worse, at a realistic base rate (few fakes among many real files) even a 'good' detector floods false alarms.
detector_acc = {"generator v1 (2023)": 0.95, "generator v2 (2026)": 0.62}   # same detector, newer generator
print("Same detector, accuracy vs generator vintage:")
for gen, acc in detector_acc.items():
    print("  {:<22} {:.0%}".format(gen, acc))
print()
# Base-rate: 10,000 media items, 1% are fake. Detector: true-positive rate 0.90, false-positive rate 0.10.
N, base_rate, tpr, fpr = 10000, 0.01, 0.90, 0.10
fakes = int(N * base_rate); real = N - fakes
tp = int(fakes * tpr); fp = int(real * fpr)
precision = tp / (tp + fp)
print("Of {:,} items, {} are fake. Detector flags {} true fakes and {} real items (TPR {:.0%}, FPR {:.0%}).".format(N, fakes, tp, fp, tpr, fpr))
print("precision = TP / (TP + FP) = {} / {} = {:.1%} -> {:.0%} of everything it flags is a FALSE alarm.".format(tp, tp + fp, precision, 1 - precision))
print("Detection decays as generators improve AND drowns in false positives at low base rates. You cannot detect your way out -")
print("the durable answer is PROVENANCE: prove what is real at the source, rather than trying to spot every fake after the fact.")

# Output:
# Same detector, accuracy vs generator vintage:
#   generator v1 (2023)    95%
#   generator v2 (2026)    62%
#
# Of 10,000 items, 100 are fake. Detector flags 90 true fakes and 990 real items (TPR 90%, FPR 10%).
# precision = TP / (TP + FP) = 90 / 1080 = 8.3% -> 92% of everything it flags is a FALSE alarm.
# Detection decays as generators improve AND drowns in false positives at low base rates. You cannot detect your way out -
# the durable answer is PROVENANCE: prove what is real at the source, rather than trying to spot every fake after the fact.

- The **same** detector scores 95 percent on a 2023 generator and 62 percent on a 2026 one — accuracy decays as generators improve.
- At a 1 percent base rate, a 90 percent / 10 percent detector flags 90 true fakes and 990 real items.
- **Precision = 90 / 1080 = about 8 percent** — 92 percent of everything it flags is a false alarm.
- Detection loses the arms race AND drowns in false positives — the durable answer is to prove what is real (provenance).

#### 💡 What just happened

⚡ What just happened? Detection fails twice: accuracy decays against better generators (95 to 62 percent), and at a realistic base rate a good detector is only ~8 percent precise (a false-alarm flood). Tradeoff: detection still helps as one layer, but it cannot be the strategy. Next: prove what is real instead of chasing what is fake.

- An accuracy bar decaying by generator vintage (95 to 62 percent)
- A base-rate confusion matrix: precision collapses to 8 percent

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Provenance (C2PA / Content Credentials)

### Provenance (C2PA / Content Credentials)

Flip the question from “is this fake?” to “can you prove it is real?” C2PA cryptographically signs media at each step - capture, edit, export - into a verifiable manifest chain. A stripped or tampered link breaks verification, and a missing manifest is itself the signal. It is the answer detection cannot be - but only if signing is widespread.

If you cannot reliably spot fakes, prove authenticity instead. **C2PA** (the Coalition for Content Provenance and Authenticity), which ships as **Content Credentials**, does exactly that: at each step in a piece of media’s life — captured by a camera, edited in software, exported, published — a cryptographically signed **manifest** records what happened and signs the previous state, forming a tamper-evident chain back to a trusted origin. To check a photo you do not analyze its pixels for artifacts; you *verify the chain of signatures*. In the worked example a signed capture-to-edit-to-export chain verifies cleanly, but when an untrusted tool swaps the edit for a deepfaked face while carrying a stale signature, the chain breaks at that link and the whole thing reads **UNVERIFIED**. This is the profound shift: provenance flips the question from “is this fake?” (unanswerable at scale) to “can you prove it is real?” (a signature check), and a *missing* manifest becomes a signal in itself — unverified content is treated with suspicion. C2PA is now embedded by Adobe, Microsoft, OpenAI (in its image outputs), Google, and camera makers, so the ecosystem is real. The one caveat: provenance only helps to the extent signing is widespread, which is why adoption is the whole game. The block validates a manifest chain and shows a tamper break, keyless.

> 📜 **Analogy**
>
> Provenance is a **notarized chain of custody for a piece of evidence**. A courtroom does not decide a document is genuine by squinting at the paper; it checks the unbroken record of custody — who created it, who handled it, each transfer signed and dated. Break the chain anywhere and the evidence is inadmissible, no matter how authentic it looks, because you can no longer *prove* its history. C2PA gives a photo or a video that same notarized trail: each step signs the last, and a verifier walks the chain back to a trusted origin. You are not judging the pixels; you are checking the paperwork — and a missing page is itself a red flag.

Detection cannot scale. How does C2PA provenance help instead?

**📝 `03_provenance_c2pa.py`** - *runnable*

In [ ]:
# PROVENANCE (C2PA / Content Credentials): instead of spotting fakes, cryptographically SIGN media at each step - capture, edit,
# export - so anyone can verify an unbroken chain back to a trusted source. A stripped or tampered step breaks the chain.
def sign(payload, prev_sig):                    # toy deterministic 'signature' (a real C2PA manifest uses public-key crypto)
    return sum(ord(c) for c in payload) * 31 + prev_sig
def verify_chain(chain):
    prev = 0
    for step in chain:
        expected = sign(step["payload"], prev)
        if step["sig"] != expected:
            return False, step["by"]
        prev = expected
    return True, None
GOOD = [{"by": "camera",   "payload": "capture:raw",  "sig": None},
        {"by": "editor",   "payload": "edit:crop",    "sig": None},
        {"by": "exporter", "payload": "export:jpeg",  "sig": None}]
prev = 0                                         # build a valid manifest chain
for s in GOOD:
    s["sig"] = sign(s["payload"], prev); prev = s["sig"]
ok, _ = verify_chain(GOOD)
print("Signed capture->edit->export chain: {}".format("VERIFIED (provenance intact)" if ok else "UNVERIFIED"))
TAMPERED = [dict(s) for s in GOOD]
TAMPERED[1] = {"by": "unknown tool", "payload": "edit:deepfake-face", "sig": TAMPERED[1]["sig"]}  # swapped content, old sig
ok2, broke = verify_chain(TAMPERED)
print("After an untrusted tool swaps the edit: {} (chain breaks at '{}')".format("VERIFIED" if ok2 else "UNVERIFIED", broke))
print()
print("Provenance flips the question from 'is this fake?' to 'can you prove it is real?'. C2PA / Content Credentials signs the")
print("chain; a missing or broken manifest is itself the signal. It is the answer detection cannot be - but only if signing is widespread.")

# Output:
# Signed capture->edit->export chain: VERIFIED (provenance intact)
# After an untrusted tool swaps the edit: UNVERIFIED (chain breaks at 'unknown tool')
#
# Provenance flips the question from 'is this fake?' to 'can you prove it is real?'. C2PA / Content Credentials signs the
# chain; a missing or broken manifest is itself the signal. It is the answer detection cannot be - but only if signing is widespread.

- Each step (capture, edit, export) is signed against the previous one, forming a verifiable **manifest chain**.
- The clean chain verifies: **VERIFIED** — provenance intact back to a trusted origin.
- When an untrusted tool swaps the edit with a stale signature, the chain breaks: **UNVERIFIED**.
- Provenance flips “is this fake?” to “can you prove it is real?” — a missing manifest is itself the signal (real C2PA uses public-key crypto).

#### 💡 What just happened

⚡ What just happened? Provenance (C2PA / Content Credentials) signs media at each step into a tamper-evident chain; you verify authenticity instead of chasing fakes, and a broken or missing manifest is the signal. Tradeoff: it only helps once signing is widespread, so adoption is the whole game. Next: for content you generate, embed a signal directly - watermarking.

- A signed capture → edit → export manifest chain
- A tampered link turns the chain from VERIFIED to UNVERIFIED

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Watermarking

### Watermarking

A statistical watermark embeds an imperceptible signal in generated content. For text, a green-list scheme (SynthID-style) biases the model toward a pseudo-random subset of the vocabulary; human text hits it at the base rate, watermarked text far more, and a z-test detects the excess (z=7.45 vs 0.30). The catch is robustness: paraphrasing rewrites tokens and drops z below threshold (2.09) - detectable, not tamper-proof.

Provenance proves the origin of media that carries a manifest; **watermarking** embeds a signal directly into the content you generate, so it can be recognized even without a manifest. For text, the standard approach (Kirchenbauer et al.; Google’s SynthID-Text) is a **green-list** scheme: at each step the model pseudo-randomly designates a subset of the vocabulary — say a quarter of it — as “green” and biases generation toward those tokens. A human writer hits green tokens at the base rate (a quarter of the time); watermarked text hits them far more often, and a **z-test** measures how many standard deviations above chance the green count sits. In the worked example, a 60-token watermarked output has 40 green tokens, giving **z = 7.45** — overwhelming evidence, far above the detection threshold of 4 — while human text with 16 green tokens gives z = 0.30 (nothing). The watermark is invisible to a reader but statistically loud. The catch is **robustness**: *paraphrasing* the watermarked text rewrites enough tokens that the green count falls to 22, dropping z to **2.09** — below threshold, watermark removed. So a watermark is a strong signal on unmodified output and a weak one after an adversary edits it: useful, but not tamper-proof, and best combined with provenance rather than relied on alone. The block runs the z-test on all three cases, keyless.

> 💧 **Analogy**
>
> A watermark is the **invisible mark on a banknote** — the thread and the UV pattern you cannot see under normal light but that glows the instant you check it the right way. It does not shout; it hides in plain sight and confirms authenticity on demand. A text watermark is the statistical equivalent: nothing a reader would notice, but a z-test lights it up like the note under a UV lamp. And it shares the banknote’s weakness — run the bill through the wash enough times, or here, paraphrase the text enough, and the mark fades below the point where the lamp can find it. A watermark proves “this came from us” on an untouched sample; it does not survive a determined launderer.

A green-list watermark makes a generated text score z=7.45, far above the z=4 threshold. Is the watermark reliable?

**📝 `04_watermarking.py`** - *runnable*

In [ ]:
# WATERMARKING generated text (statistical / green-list, as in SynthID-style schemes): at each step the model prefers a pseudo-
# random 'green' subset of the vocabulary. Human text hits green at the base rate; watermarked text hits it far more - a z-test detects it.
import math
GAMMA = 0.25                                     # green list is a quarter of the vocabulary
def z_score(green, n):                           # how many standard deviations above the chance rate
    expected = n * GAMMA
    sd = math.sqrt(n * GAMMA * (1 - GAMMA))
    return round((green - expected) / sd, 2)
N = 60; THRESH = 4.0                             # z above 4 is overwhelming evidence of a watermark
cases = [("watermarked output", 40), ("human-written text", 16), ("watermarked, then paraphrased", 22)]
for label, green in cases:
    z = z_score(green, N)
    verdict = "WATERMARK detected" if z >= THRESH else "no watermark"
    print("{:<32} {}/{} green tokens  z={:>5}  -> {}".format(label, green, N, z, verdict))
print()
print("The watermark is invisible to a reader but statistically loud (z=7.45 vs 0.30 for human text). The catch: paraphrasing")
print("rewrites enough tokens to drop z below the threshold, so a watermark is a strong signal on unmodified output, not tamper-proof.")

# Output:
# watermarked output               40/60 green tokens  z= 7.45  -> WATERMARK detected
# human-written text               16/60 green tokens  z=  0.3  -> no watermark
# watermarked, then paraphrased    22/60 green tokens  z= 2.09  -> no watermark
#
# The watermark is invisible to a reader but statistically loud (z=7.45 vs 0.30 for human text). The catch: paraphrasing
# rewrites enough tokens to drop z below the threshold, so a watermark is a strong signal on unmodified output, not tamper-proof.

- The green-list scheme biases the model toward a pseudo-random quarter of the vocabulary; a **z-test** measures the excess.
- Watermarked output: 40 of 60 green — **z = 7.45**, far above the threshold of 4 — watermark detected.
- Human text: 16 of 60 green — z = 0.30 — no watermark (green at the base rate).
- Paraphrased watermarked text: 22 of 60 green — **z = 2.09**, below threshold — the watermark is removed. Detectable, not tamper-proof.

#### 💡 What just happened

⚡ What just happened? A green-list watermark is invisible to a reader but statistically loud (z=7.45 vs 0.30 for human text); it is a strong signal on unmodified output. Tradeoff: paraphrasing rewrites tokens and drops z below threshold (2.09), so a watermark is removable - pair it with provenance. Next: decide which misuses to spend your effort on.

- Green and grey tokens with a z-score meter and a detection line
- Watermarked z=7.45, human z=0.30, paraphrased z=2.09 removed

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: The Misuse Risk Matrix

### The Misuse Risk Matrix

Turn “could this be misused?” into a ranked list. Score each harm by LIKELIHOOD (how easy or probable) and SEVERITY (how bad), multiply for a risk score, and mitigate everything above a threshold. The fake-CEO voice scam (likelihood 4 x severity 5 = 20) outranks defamation (9); four of six harms cross the threshold. Score first, then spend effort where the risk is highest.

You have a threat surface (Step 1); now prioritize it, because you cannot mitigate everything equally before launch. A **misuse risk matrix** scores each harm on two axes — **likelihood** (how easy and probable the misuse is) and **severity** (how much harm it does) — each on a small scale, then multiplies them into a single **risk score**. Everything above a threshold must have a mitigation before release; everything below is monitored. In the worked example, voice-clone fraud scores likelihood 4 times severity 5 = 20, political disinformation 16, non-consensual imagery 15, and impersonation 12, all at or above the threshold of 12 and flagged **MITIGATE**; defamation (9) and fabricated evidence (8) fall below and are monitored — so four of six harms need a control to ship. This is the same likelihood-times-severity method a safety engineer uses for any hazard, and it does two things at once: it forces you to *enumerate* the misuses honestly, and it *ranks* them so your limited pre-launch effort goes where the risk is highest rather than where the loudest voice points. The frontier labs formalize this — OpenAI’s Preparedness Framework, Anthropic’s Responsible Scaling Policy, and Google DeepMind’s Frontier Safety Framework all run structured misuse assessments before release. The block scores the matrix and counts what must be mitigated, keyless.

> 📋 **Analogy**
>
> The risk matrix is a **pre-flight risk survey**. A pilot does not treat every conceivable failure with equal alarm; they run a checklist that weighs how *likely* each problem is against how *bad* it would be, and the combination decides what gets a mandatory fix, what gets a backup, and what gets a note. An engine fire (rare but catastrophic) and a flickering cabin light (common but harmless) sit at opposite corners of the same grid, and the grid tells you where to spend the pre-flight minutes. A misuse matrix is that survey for your model: likelihood times severity turns a vague worry into a ranked list, so you fix the fake-CEO scam before you fix the typo generator.

You have six identified misuse risks and limited time before launch. How do you decide which to mitigate?

**📝 `05_misuse_risk_matrix.py`** - *runnable*

In [ ]:
# THE MISUSE RISK MATRIX: before you ship, score each harm by LIKELIHOOD (how easy/probable) and SEVERITY (how bad), multiply
# for a risk score, and mitigate everything above a threshold. This turns 'could this be misused?' into a ranked, actionable list.
RISKS = [  # (harm, likelihood 1-5, severity 1-5)
    ("voice-clone fraud",       4, 5),
    ("political disinformation", 4, 4),
    ("non-consensual imagery",   3, 5),
    ("impersonation / KYC bypass", 3, 4),
    ("fabricated evidence",      2, 4),
    ("reputation / defamation",  3, 3)]
THRESHOLD = 12                                    # risk >= 12 must be mitigated before release
scored = sorted(((L * S, h, L, S) for h, L, S in RISKS), reverse=True)
print("Misuse risk matrix (risk = likelihood x severity, threshold {}):".format(THRESHOLD))
must_mitigate = 0
for risk, h, L, S in scored:
    tag = "MITIGATE" if risk >= THRESHOLD else "monitor"
    if risk >= THRESHOLD: must_mitigate += 1
    print("  {:<28} L{} x S{} = {:>2}   {}".format(h, L, S, risk, tag))
print()
print("{} of {} harms cross the mitigation threshold and cannot ship without a control.".format(must_mitigate, len(RISKS)))
print("Likelihood x severity ranks where to spend effort: the fake-CEO voice scam (20) outranks defamation (9). Score, then mitigate.")

# Output:
# Misuse risk matrix (risk = likelihood x severity, threshold 12):
#   voice-clone fraud            L4 x S5 = 20   MITIGATE
#   political disinformation     L4 x S4 = 16   MITIGATE
#   non-consensual imagery       L3 x S5 = 15   MITIGATE
#   impersonation / KYC bypass   L3 x S4 = 12   MITIGATE
#   reputation / defamation      L3 x S3 =  9   monitor
#   fabricated evidence          L2 x S4 =  8   monitor
#
# 4 of 6 harms cross the mitigation threshold and cannot ship without a control.
# Likelihood x severity ranks where to spend effort: the fake-CEO voice scam (20) outranks defamation (9). Score, then mitigate.

- Each harm is scored on **likelihood** and **severity** (1–5 each); risk = likelihood times severity.
- Voice-clone fraud (4 x 5 = 20) tops the list, then disinformation (16), non-consensual imagery (15), impersonation (12).
- The threshold is 12: **4 of 6** harms cross it and must be mitigated before shipping; two fall below and are monitored.
- The matrix enumerates AND ranks the misuses, so limited pre-launch effort goes where the risk is highest.

#### 💡 What just happened

⚡ What just happened? A misuse risk matrix scores each harm by likelihood times severity and mitigates everything above a threshold (4 of 6 here), ranking effort by risk. Tradeoff: the scores are judgement calls, best set with a red team - engineers who adversarially probe your own model before launch, trying to coerce its safety filters into producing the mapped harms, so the likelihood scores reflect what an attacker can actually achieve rather than a guess. Next: for the harms you must mitigate, no single guardrail is enough - so you layer them.

- A likelihood-by-severity grid plotting six harms
- Cells at or above the threshold of 12 flag MITIGATE (4 of 6)

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Defense-in-Depth

### Defense-in-Depth

No single guardrail stops misuse, so you stack independent layers - input filter, output classifier, provenance/watermark, rate-limit/KYC, human review. An abusive request slips through only if EVERY layer misses it, so the miss rates MULTIPLY: five leaky layers (letting 25 to 60 percent through each) combine to a 0.009 miss - the stack catches 99.1 percent, though the best single layer catches only 75 percent. The Swiss-cheese model.

The harms you flagged to mitigate need controls, and the mistake is to pin your hopes on one perfect filter — there is no such thing. **Defense-in-depth** instead stacks several *independent* guardrails so that an abusive request has to defeat all of them. In the worked example the layers are an input prompt filter (which carries the consent and identity check from Step 1 — refusing to synthesize a specific real person without authorization), an output content classifier, provenance and watermarking on outputs, rate-limiting plus KYC on high-risk features, and human review of flagged cases — and each one is leaky, letting between 25 and 60 percent of abuse through on its own. The crucial arithmetic is that a request slips all the way through *only if every layer misses it*, so for independent layers the combined miss rate is the **product** of the individual miss rates: 0.40 times 0.30 times 0.50 times 0.60 times 0.25 = **0.009**. That is a **99.1 percent catch rate** from five mediocre filters — even though the best single layer catches only 75 percent. This is the same multiplication you saw for reliability in Module 12, now pointed at abuse, and it is why “the filter isn’t perfect” is never a reason to skip a layer: imperfect independent layers compound into a strong net. The two cautions are that the layers must be genuinely independent (a shared blind spot breaks the multiplication) and that no stack reaches 100 percent, so you still need the disclosure and takedown path of the final step. The block multiplies the miss rates, keyless.

> 🧀 **Analogy**
>
> Defense-in-depth is the **Swiss-cheese model** of safety. Each slice of cheese — each guardrail — has holes, and any one slice will let some things through. But stack several slices and line them up, and a hole in one is almost always backed by solid cheese in the next; a threat only passes if the holes happen to line up all the way through, which is rare. The trick is that the slices must be *different* cheeses — if every layer has its hole in the same spot (a shared blind spot), stacking them adds nothing. Five leaky filters that fail in different ways make one net that almost never leaks; five filters that fail the same way are just one filter drawn five times.

Each of your five guardrails individually lets 25 to 60 percent of abuse through. Is layering them hopeless?

**📝 `06_defense_in_depth.py`** - *runnable*

In [ ]:
# DEFENSE-IN-DEPTH: no single guardrail stops misuse, so you stack independent layers. An abusive request slips through only if
# EVERY layer misses it, so the miss rates MULTIPLY - even mediocre layers combine into a strong net. (The Swiss-cheese model.)
LAYERS = [  # (layer, miss rate = fraction of abuse it lets through)
    ("input prompt filter",          0.40),
    ("output content classifier",    0.30),
    ("provenance + watermark",       0.50),
    ("rate-limit + KYC on high-risk", 0.60),
    ("human review of flagged",      0.25)]
combined_miss = 1.0
print("Layered defenses (an abusive request must beat ALL of them):")
for name, miss in LAYERS:
    print("  {:<32} lets through {:.0%}".format(name, miss))
    combined_miss *= miss
best_single = min(m for _, m in LAYERS)
print()
print("best SINGLE layer catches {:.0%} - not enough on its own.".format(1 - best_single))
print("STACKED, misses multiply: combined miss = {:.3f} -> the stack catches {:.1%} of abuse.".format(combined_miss, 1 - combined_miss))
print("No guardrail is perfect, so layer them: independent defenses turn several leaky filters into one that rarely leaks.")

# Output:
# Layered defenses (an abusive request must beat ALL of them):
#   input prompt filter              lets through 40%
#   output content classifier        lets through 30%
#   provenance + watermark           lets through 50%
#   rate-limit + KYC on high-risk    lets through 60%
#   human review of flagged          lets through 25%
#
# best SINGLE layer catches 75% - not enough on its own.
# STACKED, misses multiply: combined miss = 0.009 -> the stack catches 99.1% of abuse.
# No guardrail is perfect, so layer them: independent defenses turn several leaky filters into one that rarely leaks.

- Five independent guardrails, each leaky — letting between 25 and 60 percent of abuse through on its own.
- A request slips through **only if every layer misses**, so the combined miss is the product: 0.40 x 0.30 x 0.50 x 0.60 x 0.25.
- Combined miss = **0.009** — the stack catches **99.1 percent**, though the best single layer catches only 75 percent.
- No guardrail is perfect, so layer them (independently) — and no stack hits 100 percent, so keep the disclosure and takedown path.

#### 💡 What just happened

⚡ What just happened? No single guardrail suffices, so you stack independent layers whose miss rates multiply: five leaky filters combine to a 0.009 miss (99.1 percent caught) versus 75 percent for the best single layer. Tradeoff: the layers must be genuinely independent, and no stack reaches 100 percent. Next: the law now requires you to disclose what is AI-generated - and be ready to respond.

- Five stacked Swiss-cheese guardrail layers
- Misses multiply to 0.009 → the stack catches 99.1 percent

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: Disclosure & the Incident Gate

### Disclosure & the Incident Gate

Transparency is now law. The EU AI Act (Article 50) requires AI-generated content and deepfakes to be LABELLED, and a responsible provider needs a takedown-and-notify process BEFORE an incident, not after. Gate the release on five checks - provenance/watermark on outputs, a disclosure label, a takedown process, a reporting channel, red-team sign-off - and one unmet check (the label) BLOCKS the ship.

The final step turns the toolkit into a release gate, and adds the obligation the others do not: **disclosure**. As of 2026, the **EU AI Act Article 50** transparency rules require that AI-generated content and deepfakes be **labelled** so people know they are interacting with synthetic media — this is law, not courtesy. So the gate mirrors both the engineering and the legal duties: do the outputs carry provenance or a watermark, is AI-generated content disclosed and labelled, is there a takedown-and-victim-notification process, is a misuse reporting channel live, and has a red team signed off on the misuse risks? In the worked example four checks pass but one — the disclosure label — is unmet, so the gate **BLOCKS** the release: shipping without the label is an Article 50 non-compliance. The other half of this step is *incident readiness*: the takedown-and-notify path has to exist *before* an incident, because the moment a victim reports a non-consensual deepfake or a fraud, you need a way to pull the content and notify affected people fast — improvising it during the incident is how a bad situation becomes a catastrophe. Transparency plus a response plan is the release bar for any synthetic-media feature. The block evaluates the gate, keyless.

> 🏷️ **Analogy**
>
> The disclosure gate is a **“contains AI” label, the way food carries a “contains nuts” warning**. Society decided long ago that some facts are too important to leave to the buyer’s inspection — you do not make an allergic person chemically analyze a cookie; you require the maker to label it. Synthetic media crossed the same line: people cannot be expected to forensically test every image and voice, so the maker must disclose. And like a food recall, the label is only half the system — you also need the process to pull a bad batch and notify the people affected, ready before anything goes wrong. Ship the label and the recall plan together, or do not ship.

Your synthetic-media feature is ready but does not label AI-generated outputs. Can you ship it in the EU?

**📝 `07_disclosure_gate.py`** - *runnable*

In [ ]:
# DISCLOSURE + THE INCIDENT GATE (EU AI Act Art. 50): 2026 transparency law requires that AI-generated content and deepfakes be
# LABELLED, and that you can respond when abuse happens. Gate the release on disclosure, provenance, takedown, reporting, and sign-off.
CHECKS = {                                        # each must hold to ship a synthetic-media feature
    "outputs carry provenance / watermark": True,
    "AI-generated content is disclosed / labelled (EU AI Act Art. 50)": False,   # UNMET
    "takedown + victim-notification process exists": True,
    "misuse reporting channel is live": True,
    "red-team sign-off on misuse risks": True}
unmet = [name for name, ok in CHECKS.items() if not ok]
print("SYNTHETIC-MEDIA RELEASE GATE:")
for name, ok in CHECKS.items():
    print("  [{}] {}".format("x" if ok else " ", name))
print()
print("gate: {}".format("PASS - ship" if not unmet else "BLOCK - {} unmet:".format(len(unmet))))
for u in unmet:
    print("   - " + u)
print("Disclosure is now law, not courtesy: the EU AI Act (Art. 50) requires labelling AI-generated media and deepfakes, and a")
print("provider needs a takedown-and-notify path BEFORE an incident, not after. Transparency plus a response plan is the release bar.")

# Output:
# SYNTHETIC-MEDIA RELEASE GATE:
#   [x] outputs carry provenance / watermark
#   [ ] AI-generated content is disclosed / labelled (EU AI Act Art. 50)
#   [x] takedown + victim-notification process exists
#   [x] misuse reporting channel is live
#   [x] red-team sign-off on misuse risks
#
# gate: BLOCK - 1 unmet:
#    - AI-generated content is disclosed / labelled (EU AI Act Art. 50)
# Disclosure is now law, not courtesy: the EU AI Act (Art. 50) requires labelling AI-generated media and deepfakes, and a
# provider needs a takedown-and-notify path BEFORE an incident, not after. Transparency plus a response plan is the release bar.

- The gate mirrors the legal and engineering duties: provenance/watermark, a disclosure label, a takedown process, a reporting channel, red-team sign-off.
- Four checks pass but one — the **AI-generated disclosure label** — is unmet, so the gate **BLOCKS** the release.
- Shipping without the label is an **EU AI Act Article 50** non-compliance — disclosure is now law.
- The takedown-and-notify path must exist **before** an incident, not after — transparency plus a response plan is the release bar.

#### 💡 What just happened

⚡ What just happened? Disclosure is law: EU AI Act Article 50 requires labelling AI-generated media and deepfakes, and the gate BLOCKS on the unmet label - plus a takedown-and-notify path must exist before an incident. Tradeoff: the gate can block a launch on a labelling gap, which is the point - it fails safe. That is the whole toolkit: map the surface, know detection loses, prove provenance, watermark, score misuse, layer defenses, and disclose.

> 📦 **Info**
>
> 🚫 Two things this toolkit does not do for youThe defenses are necessary, not sufficient, and this is not legal advice. First, **no control is complete**: detection loses, watermarks wash out, provenance depends on adoption, and a defense stack never reaches 100 percent — so the honest posture is layered risk reduction against a stated threat model, not “solved.” Second, **a green gate is not a compliance sign-off**: the EU AI Act Article 50 label is one obligation among many (prohibited practices, high-risk duties, sector deepfake laws like the US TAKE IT DOWN Act on non-consensual imagery), and which apply to you is a legal determination for your counsel and covered in Lesson 15.6. The engineer’s job is to make the misuse visible, the outputs verifiable, and the easy failures impossible — and to hand the residual-risk and jurisdiction calls to the people who own them.

- A five-item release gate with one unmet check
- The unmet disclosure label BLOCKS the ship (EU AI Act Art. 50)

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture — defending synthetic media
> Deepfake defense is a pipeline, and its first principle is counter-intuitive: stop trying to catch fakes and start proving what is real. **Map the threat surface** — your capabilities define which harms you enable (Step 1). **Know detection loses** — to the arms race and the base-rate flood (Step 2). **Prove provenance** — sign media into a verifiable chain, C2PA (Step 3). **Watermark** — embed a signal in your outputs, knowing it can be washed out (Step 4). **Score misuse** — likelihood times severity, mitigate above the line (Step 5). **Layer defenses** — independent guardrails whose misses multiply (Step 6). And **disclose** — label AI-generated content and keep a takedown path ready, because Article 50 requires it (Step 7). Ask two questions before you ship a generative feature: have I made my outputs *verifiable*, and do I know — with a ranked list — which misuses I enable and how each is mitigated?

**📦 **The synthetic-media defense picker****

| Defense | What it does | The guarantee | The failure mode |
|---|---|---|---|
| Detection | Flags synthetic media after the fact | Weak - a decaying classifier | Arms race + false-positive flood |
| Provenance (C2PA) | Proves what is real at the source | Verifiable signed chain of custody | Only works where signing is adopted |
| Watermarking | Embeds a signal in your outputs | Strong on unmodified content | Paraphrase / re-encode removes it |
| Disclosure | Labels content as AI-generated | Legal transparency (Art. 50) | Relies on the platform honoring it |

> 📦 **Info**
>
> ➡️ Where this goes nextYou can now ship a generative feature with the misuse thought through — mapped, ranked, made verifiable, layered, and disclosed. This builds on privacy from Lesson 15.4: a deepfake is the misuse of a person’s synthetic identity, where privacy is the leak of their real data. The full regime that turns disclosure and misuse controls into legal obligations — the EU AI Act risk tiers, transparency, and prohibited practices — is the regulatory landscape in Lesson 15.6, and the governance program that owns the misuse assessment, the red-team sign-off, and the incident process closes the module in Lesson 15.7. The production frontier is [C2PA](https://github.com/c2pa-org/c2pa-specifications) and [SynthID](https://www.nature.com/articles/s41586-024-08025-4) plus the frontier-lab preparedness frameworks; the governing text is the [EU AI Act Article 50](https://artificialintelligenceact.eu/article/50/). This is defensive engineering education, not legal advice.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Deepfakes & Misuse— Defending Synthetic Media**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-15_5.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-15.5.html` - regenerate this notebook whenever the source HTML is updated.*
